# Setup

In [1]:
!uv pip install torch>=2.0.0
!uv pip install transformers>=4.30.0
!uv pip install datasets>=2.10.0
!uv pip install peft>=0.4.0
!uv pip install accelerate>=0.20.0
!uv pip install bitsandbytes>=0.39.0
!uv pip install PyYAML>=6.0
!uv pip install numpy>=1.21.0
!uv pip install tqdm>=4.64.0
!uv pip install unsloth[colab-new]
!uv pip install xformers

Using Python 3.12.13 environment at: /usr
Checked 1 package in 346ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 238ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 211ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 187ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 184ms
Using Python 3.12.13 environment at: /usr
Resolved 31 packages in 425ms
Prepared 1 package in 1.59s
Installed 1 package in 17ms
 + bitsandbytes==0.49.2
Using Python 3.12.13 environment at: /usr
Checked 1 package in 252ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 304ms
Using Python 3.12.13 environment at: /usr
Checked 1 package in 345ms
Using Python 3.12.13 environment at: /usr
Resolved 101 packages in 2.08s
Prepared 12 packages in 3.36s
Uninstalled 4 packages in 725ms
Installed 12 packages in 172ms
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + hf-transfer==0.1.9
 + msgspec==0.21.1
 - pyarrow==18.1.0
 + 

In [2]:
!uv pip install unsloth
from unsloth import FastLanguageModel

Using Python 3.12.13 environment at: /usr
Checked 1 package in 237ms
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Configuration for GPT-OSS20B Cybersecurity Fine-tuning

# Model configuration
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
output_dir = "./gpt-oss-cybersecurity-lora"
max_seq_length = 1024

# Dataset configuration
validation_split = 0.1

# LoRA configuration
lora_r = 16
lora_alpha = 32
lora_dropout = 0.1

# Training configuration
batch_size = 1
eval_batch_size = 2
gradient_accumulation_steps = 16
epochs = 3
learning_rate = 0.0002
logging_steps = 10
eval_steps = 100
save_steps = 500
warmup_steps = 100

# Hardware considerations:
# - Reduce batch_size if you have memory issues
# - Increase gradient_accumulation_steps to maintain effective batch size
# - Consider reducing max_length for memory constraints
# - Use load_in_4bit instead of load_in_8bit for even lower memory usage

# Training

In [4]:
#!/usr/bin/env python3
"""
Fine-tuning Llama 3.2 for Cybersecurity with Unsloth
This script implements Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA)
to fine-tune a Llama 3.2 model for cybersecurity applications.
Requirements:
- CUDA-capable GPU with at least 16GB VRAM (recommended)
- Python 3.8+
- See requirements.txt for package dependencies
Usage:
    python fine_tune_gpt_oss_cybersecurity.py --config config.yaml
Date: 2025-08-10
"""

import argparse
import json
import logging
import os
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from datasets import Dataset, load_dataset, concatenate_datasets
from transformers import AutoTokenizer, TrainingArguments
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import yaml



In [5]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [6]:
llama3_template = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""

def format_prompt_llama3(example: Dict) -> Dict:
    """Formats an example for Llama-3.2-Instruct fine-tuning."""
    instruction = example.get('instruction') or example.get('question') or example.get('input') or ""
    response = example.get('response') or example.get('answer') or example.get('output') or ""
    return {"text": llama3_template.format(instruction, response)}

class CybersecurityFineTuner:
    def __init__(self, config: Dict):
        self.config = config
        self.model_name = config.get('model_name', 'unsloth/Llama-3.2-3B-Instruct-bnb-4bit')
        self.output_dir = config.get('output_dir', './llama3-cybersecurity-lora')
        self.max_length = config.get('max_length', 1024)
        self.tokenizer = None
        self.model = None
        self.train_dataset = None
        self.eval_dataset = None

    def load_model(self) -> None:
        logger.info(f"Loading model from {self.model_name}")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.model_name,
            max_seq_length=self.max_length,
            dtype=None,
            load_in_4bit=True,
        )
        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.config.get('lora_r', 16),
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=self.config.get('lora_alpha', 32),
            lora_dropout=self.config.get('lora_dropout', 0.1),
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
        )

    def load_and_prepare_datasets(self, downsample_size: int = 5000) -> None:
        dataset_names = [
            "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset",
            "Cogensec/OptikalLLM_10k_dataset",
            "Mohannadcse/cybersec-reasoning-merged"
        ]
        logger.info(f"Loading datasets: {', '.join(dataset_names)}")
        datasets = [load_dataset(name, split='train') for name in dataset_names]

        def standardize(example):
            # Robustly find instruction and response keys
            instr = example.get('instruction') or example.get('prompt') or ""
            resp = example.get('response') or example.get('output') or ""
            return {"instruction": instr, "response": resp}

        standardized_datasets = []
        for ds in datasets:
            standardized_datasets.append(ds.map(standardize, remove_columns=ds.column_names))

        merged_dataset = concatenate_datasets(standardized_datasets)
        if downsample_size > 0 and len(merged_dataset) > downsample_size:
            merged_dataset = merged_dataset.shuffle(seed=42).select(range(downsample_size))

        self.dataset = merged_dataset.map(format_prompt_llama3)
        if self.config.get('validation_split', 0.1) > 0:
            split = self.dataset.train_test_split(test_size=self.config.get('validation_split', 0.1))
            self.train_dataset, self.eval_dataset = split["train"], split["test"]
        else:
            self.train_dataset, self.eval_dataset = self.dataset, None

    def train(self):
        self.load_model()
        self.load_and_prepare_datasets(downsample_size=self.config.get('downsample_size', 5000))
        trainer = UnslothTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            args=UnslothTrainingArguments(
                per_device_train_batch_size=self.config.get('batch_size', 2),
                gradient_accumulation_steps=self.config.get('gradient_accumulation_steps', 8),
                warmup_steps=self.config.get('warmup_steps', 100),
                num_train_epochs=self.config.get('epochs', 3),
                learning_rate=self.config.get('learning_rate', 2e-4),
                fp16=not torch.cuda.is_bf16_supported(),
                bf16=torch.cuda.is_bf16_supported(),
                logging_steps=self.config.get('logging_steps', 1),
                optim="adamw_8bit",
                output_dir=self.output_dir,
            ),
        )
        trainer.train()

In [7]:
def main():
    """Main function adapted for notebook execution using global variables."""
    # Build configuration from global variables defined in the notebook
    config = {
        "model_name": model_name,
        "output_dir": output_dir,
        "max_length": max_seq_length,
        "validation_split": validation_split,
        "lora_r": lora_r,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "logging_steps": logging_steps,
        "warmup_steps": warmup_steps,
        "downsample_size": 5000 # Default for safety, can be moved to config cell if needed
    }

    logger.info("Starting training with notebook-defined configuration")

    fine_tuner = CybersecurityFineTuner(config)
    fine_tuner.train()

if __name__ == "__main__":
    main()

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,4.269565
20,3.772210
30,2.758919
40,2.214832
50,1.985221
60,1.636394
70,1.563986
80,1.386955
90,1.121363
100,1.126825


Unsloth: Restored added_tokens_decoder metadata in ./gpt-oss-cybersecurity-lora/checkpoint-500/tokenizer_config.json.


Step,Training Loss
10,4.269565
20,3.772210
30,2.758919
40,2.214832
50,1.985221
60,1.636394
70,1.563986
80,1.386955
90,1.121363
100,1.126825


Unsloth: Restored added_tokens_decoder metadata in ./gpt-oss-cybersecurity-lora/checkpoint-846/tokenizer_config.json.


# Inferencing

In [14]:
import torch
from unsloth import FastLanguageModel
import os

# --- STEP 1: LOAD MODEL & ADAPTERS ---
# This only needs to run once per session
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

checkpoint_dirs = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-') and os.path.isdir(os.path.join(output_dir, d))]
if not checkpoint_dirs:
    raise FileNotFoundError(f"No checkpoint directories found in {output_dir}.")

latest_checkpoint_name = sorted(checkpoint_dirs, key=lambda x: int(x.split('-')[1]))[-1]
latest_checkpoint_path = os.path.join(output_dir, latest_checkpoint_name)

print(f"Loading adapters from: {latest_checkpoint_path}")
model.load_adapter(latest_checkpoint_path)
print("Model and adapters loaded successfully.")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Loading adapters from: ./gpt-oss-cybersecurity-lora/checkpoint-846


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Model and adapters loaded successfully.


In [21]:
# --- STEP 2: RUN INFERENCE ---
prompt_text = "Explain what an SQL injection attack is and how to prevent it."
formatted_prompt = llama3_template.format(prompt_text, "")

inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    use_cache=True,
    do_sample=True,
    temperature=1,
    top_k=50,
    top_p=0.95,
    eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")],
)

print("Inference complete. Run the next cell to see the result.")

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Inference complete. Run the next cell to see the result.


In [23]:
# --- STEP 3: DECODE & DEBUG ---
# print(f"\n--- Raw Model Outputs ---")
# print(outputs)

response = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

# Extraction logic: Look for the LAST occurrence of the assistant header
# to skip any echoes or template duplicates.
start_marker = "<|start_header_id|>assistant<|end_header_id|>"
end_marker = "<|eot_id|>"

# Find the last start marker
start_index = response.rfind(start_marker)

if start_index != -1:
    # Move index past the marker and any trailing newlines
    content_start = start_index + len(start_marker)

    # Find the end marker specifically after the start content
    end_index = response.find(end_marker, content_start)

    if end_index != -1:
        extracted_response = response[content_start:end_index].strip()
    else:
        extracted_response = response[content_start:].strip()
else:
    extracted_response = f"Could not parse assistant's response. Full text:\n{response}"

print(f"\n--- Inference Result ---")
print(f"Prompt: {prompt_text}")
print(f"Generated Response: {extracted_response}")


--- Inference Result ---
Prompt: Explain what an SQL injection attack is and how to prevent it.
Generated Response: SQL injection occurs when users input malformed queries via form inputs, allowing attackers to execute arbitrary code on the database. To prevent it, validate user inputs as SQL, use parameterized queries, and restrict SQL dialect access.


### Export to GGUF for Ollama
To use this model in Ollama, we need to export it to GGUF format. You can choose different quantization methods (e.g., `q4_k_m` is a good balance of speed and quality).

In [25]:
# Re-running GGUF export after ensuring config.json exists
from unsloth import FastLanguageModel
import os

# 1. Ensure we are targeting the correct latest checkpoint
checkpoint_dirs = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
latest_checkpoint = os.path.join(output_dir, sorted(checkpoint_dirs, key=lambda x: int(x.split('-')[1]))[-1])

# 2. Load model + adapter correctly
# This time it should detect the PEFT/LoRA weights properly
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = latest_checkpoint,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3. Export to GGUF
# Using q4_k_m for a good balance of size and performance
model.save_pretrained_gguf(
    "model_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

print(f"Success! Your fine-tuned model from {latest_checkpoint} has been exported to the 'model_gguf' folder.")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:57<00:00, 28.82s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:39<00:00, 49.74s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf_gguf/Llama-3.2-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root

In [26]:
from google.colab import files
import os

# Path to the exported GGUF file
gguf_path = "model_gguf_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf"

if os.path.exists(gguf_path):
    print(f"Downloading {gguf_path}...")
    files.download(gguf_path)
else:
    print("GGUF file not found. Please check the sidebar for the correct path.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Fix: Missing config.json

In [ ]:
import os
import shutil
from transformers import AutoConfig

# 1. Check if config.json exists in the checkpoint directory
config_path = os.path.join(latest_checkpoint, 'config.json')

if not os.path.exists(config_path):
    print(f"[!] config.json missing in {latest_checkpoint}. Copying from base model...")
    # Load config from the base model name used in training
    config = AutoConfig.from_pretrained(model_name)
    config.save_pretrained(latest_checkpoint)
    print("[+] config.json has been created in the checkpoint directory.")
else:
    print(f"[+] config.json already exists in {latest_checkpoint}.")

# 2. List directory to verify
print(f"\nContents of {latest_checkpoint}:")
print(os.listdir(latest_checkpoint))

### Re-try GGUF Export
Once the configuration file is verified/fixed above, run the export cell again. The `save_pretrained_gguf` method requires the `config.json` to properly map the architecture for the GGUF file.